# Assignment 4: Phrase Queries and PageRank

**Student names**: Eskild Sæther <br>
**Group number**: 100 <br>
**Date**: 25.10.2026

## Important notes
1. You may work in groups of maximum 2 students.
2. Submit in **.ipynb** format, with your code, outputs and typed answers.
3. Implement the TODO functions yourself. You may use Python’s standard library or NumPy. Do not use libraries that implement indexing or PageRank for you.

**Due date**: 30.10.2026 23:59

In this assignment, you will build indexes for phrase queries, compute PageRank, and answer conceptual questions.

## Dataset and preprocessing

Place `cran.all.1400` next to this notebook. The parser below loads its 1,400 documents into `docs`.
Use only each document's `abstract` field.

In [3]:
# Read 'cran.all.1400' and parse the documents into a suitable data structure

CRAN_PATH = r"./cran.all.1400"  # <-- change this if needed

def parse_cranfield(path):
    docs = {}
    current_id = None
    current_field = None
    buffers = {"T": [], "A": [], "B": [], "W": []}
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith(".I "):
                if current_id is not None:
                    docs[current_id] = {
                        "id": current_id,
                        "title": " ".join(buffers["T"]).strip(),
                        "abstract": " ".join(buffers["W"]).strip()
                    }
                current_id = int(line.split()[1])
                buffers = {k: [] for k in buffers}
                current_field = None
            elif line.startswith("."):
                tag = line[1:].strip()
                current_field = tag if tag in buffers else None
            else:
                if current_field is not None:
                    buffers[current_field].append(line)
    if current_id is not None:
        docs[current_id] = {
            "id": current_id,
            "title": " ".join(buffers["T"]).strip(),
            "abstract": " ".join(buffers["W"]).strip()
        }
    print(f"Parsed {len(docs)} documents.")
    return docs

docs = parse_cranfield(CRAN_PATH)

Parsed 1400 documents.


### Preprocessing

Implement `tokenize(text)` and tokenize the abstracts. You may reuse your tokenizer from an earlier assignment, but this time keep the stopwords.

In [4]:
# TODO: Implement preprocessing
import re

removal = set(['.', ',', ';', '(', ')', ':', '/', '\\'])
def tokenize(text: str) -> list[str]:
    text = text.lower()
    for char in removal:
        text = text.replace(char, "")
    
    tokens: list[str] = text.split()
    return tokens

#den lager en ny dict med doc_id som key og item er en liste med tokens.
token_lists = {doc_id: tokenize(doc["abstract"]) for doc_id, doc in docs.items()}

print(token_lists[1])

['experimental', 'investigation', 'of', 'the', 'aerodynamics', 'of', 'a', 'wing', 'in', 'a', 'slipstream', 'an', 'experimental', 'study', 'of', 'a', 'wing', 'in', 'a', 'propeller', 'slipstream', 'was', 'made', 'in', 'order', 'to', 'determine', 'the', 'spanwise', 'distribution', 'of', 'the', 'lift', 'increase', 'due', 'to', 'slipstream', 'at', 'different', 'angles', 'of', 'attack', 'of', 'the', 'wing', 'and', 'at', 'different', 'free', 'stream', 'to', 'slipstream', 'velocity', 'ratios', 'the', 'results', 'were', 'intended', 'in', 'part', 'as', 'an', 'evaluation', 'basis', 'for', 'different', 'theoretical', 'treatments', 'of', 'this', 'problem', 'the', 'comparative', 'span', 'loading', 'curves', 'together', 'with', 'supporting', 'evidence', 'showed', 'that', 'a', 'substantial', 'part', 'of', 'the', 'lift', 'increment', 'produced', 'by', 'the', 'slipstream', 'was', 'due', 'to', 'a', 'destalling', 'or', 'boundary-layer-control', 'effect', 'the', 'integrated', 'remaining', 'lift', 'incremen

## 4.1 Indexes for phrase queries

### 4.1.1 Biword index

A biword is a pair of consecutive tokens. Implement:

- `build_biword_index(token_lists)`: return a dictionary mapping each biword, written as two tokens separated by one space, to sorted, unique document IDs.
- `biword_phrase_search(phrase, index)`: tokenize the phrase and return the sorted IDs of documents containing all its biwords. You may assume at least two query tokens.

In [5]:
# TODO: Build and search the biword index
def build_biword_index(token_lists):
    #Så jeg skal ha en dict entry for alle biwords som i collectionene?
    #Er ikke det veldig mange dict entries?
    biword_index = {}
    for docid, token_list in token_lists.items():
        for i in range(0, len(token_list)-1):
            dual_token = token_list[i] + ' ' + token_list[i+1]
            #Legger til nøkkel
            if(dual_token not in biword_index.keys()):
                biword_index[dual_token] = [docid]

            elif(dual_token in biword_index.keys()):
                if(docid not in biword_index[dual_token]):
                    biword_index[dual_token].append(docid)

    return biword_index

def biword_phrase_search(phrase, index):
    tokenized_phrase = tokenize(phrase)
    biwords = []
    for i in range (0, len(tokenized_phrase)-1):
        dual_token = tokenized_phrase[i] + ' ' + tokenized_phrase[i+1]
        biwords.append(dual_token)

    #Starter med dokumentene fra den første biwordet
    document_ids = set(index[biwords[0]])

    #Tar intersection mellom docid-er som er retrieved
    for i in range(1, len(biwords)):
        document_ids = document_ids & set(index[biwords[i]])

    document_ids = list(document_ids)
    document_ids.sort()
    return document_ids

biword_index = build_biword_index(token_lists)
print(biword_index['experimental investigation'])

doc_ids = biword_phrase_search("experimental investigAtion", biword_index)
print(doc_ids)
#Med en slik biwordi index kan man ikke søke opp enkeltord, gjør ikke dette
#search indexen ubrukelig?

[1, 29, 84, 170, 173, 176, 189, 245, 251, 372, 423, 442, 497, 505, 522, 552, 569, 635, 636, 662, 694, 713, 766, 772, 801, 816, 836, 844, 858, 887, 932, 986, 996, 1019, 1039, 1074, 1092, 1097, 1098, 1156, 1159, 1161, 1205, 1220, 1225, 1227, 1230, 1231, 1338, 1364]
[1, 29, 84, 170, 173, 176, 189, 245, 251, 372, 423, 442, 497, 505, 522, 552, 569, 635, 636, 662, 694, 713, 766, 772, 801, 816, 836, 844, 858, 887, 932, 986, 996, 1019, 1039, 1074, 1092, 1097, 1098, 1156, 1159, 1161, 1205, 1220, 1225, 1227, 1230, 1231, 1338, 1364]


### 4.1.2 Positional index

Implement `build_positional_index(token_lists)` with the structure:

```python
index[term][doc_id] = [position_1, position_2, ...]
```

Store every occurrence, with positions starting at 1 within each document. Keep document IDs and position lists in ascending order.

Then implement `phrase_search(phrase, index)` using the positional index. Return sorted, unique IDs of documents containing the complete phrase.

Return `[]` for an empty query or a query token absent from the index.

In [35]:
# TODO: Build the positional index and implement phrase search
def build_positional_index(token_lists):
    positional_index = {}
    for docid, token_list in token_lists.items():
        for index, token in zip(range(0,len(token_list)), token_list):
            if(token not in positional_index.keys()):
                positional_index[token] = {}

            if(docid not in positional_index[token].keys()):
                positional_index[token][docid] = [index]

            elif(docid in positional_index[token].keys()):
                positional_index[token][docid].append(index)

    return positional_index

def phrase_search(phrase, index):
    tokenized_phrase = tokenize(phrase)

    if not tokenized_phrase or any(t not in index for t in tokenized_phrase):
        return []

    doc_ids = set(index[tokenized_phrase[0]].keys())
    for term in tokenized_phrase[1:]:
        doc_ids &= set(index[term].keys())

    result = []
    for doc_id in doc_ids:
        positions = [set(index[term][doc_id]) for term in tokenized_phrase]

        for start in positions[0]:
            if all(start + i in positions[i] for i in range(1, len(positions))):
                result.append(doc_id)
                break

    return sorted(result)

positional_index = build_positional_index(token_lists)

results = phrase_search("flight", positional_index)

### 4.1.3 Run phrase queries

Run the following queries with both indexes. The cell prints the number of returned documents and the first five IDs.

In [36]:
phrase_queries = [
    "boundary layer",
    "laminar boundary layer",
    "free stream mach number",
    "in the wind tunnel",
]

for query in phrase_queries:
    biword_docs = biword_phrase_search(query, biword_index)
    phrase_docs = phrase_search(query, positional_index)
    print(f"{query!r}")
    print(f"  Biword:     {len(biword_docs):3d} documents; first five: {biword_docs[:5]}")
    print(f"  Positional: {len(phrase_docs):3d} documents; first five: {phrase_docs[:5]}")

'boundary layer'
  Biword:     291 documents; first five: [2, 3, 4, 7, 8]
  Positional: 291 documents; first five: [2, 3, 4, 7, 8]
'laminar boundary layer'
  Biword:     105 documents; first five: [4, 9, 21, 23, 43]
  Positional:  97 documents; first five: [4, 9, 21, 23, 43]
'free stream mach number'
  Biword:       7 documents; first five: [73, 310, 470, 519, 522]
  Positional:   7 documents; first five: [73, 310, 470, 519, 522]
'in the wind tunnel'
  Biword:       3 documents; first five: [140, 141, 795]
  Positional:   2 documents; first five: [141, 795]


## 4.2 PageRank

The graph in `painters.json` represents 14 Wikipedia pages about painters and 50 directed links between them. An arrow from A to B means that page A links to page B. We will rank the pages within this collection.

![Directed links between the painters. Numbers identify names in the legend.](painters.svg)

The provided cell loads the collection and creates the adjacency matrix `A` as a list of rows. Each row identifies the page a link comes from, and each column identifies the destination page. Thus, `A[i][j] = 1` means that page `i` links to page `j`. If there is no link, the entry is 0.

1. Implement `transition_matrix(A, alpha=0.15)`. Return a new transition matrix `P` with floating-point probabilities, as a list of rows or a NumPy array, where `P[i][j]` is the probability of moving from page `i` to page `j`. Each row must sum to 1.
   - If a page has outgoing links, distribute probability `1 - alpha` equally among them, and probability `alpha` equally among **all** pages, including the current page.
   - If a page has no outgoing links (a dead end), distribute its entire probability equally among all pages.

   For example, Picasso links to Manet and Matisse. With `alpha=0.15`, split the 85% link-following probability between those two destinations, and distribute the remaining 15% across all fourteen pages.

2. Implement `pagerank(P, x0, iterations=50)`. Starting from the provided initial scores `x0`, repeatedly update the score vector using $x_{t+1}=x_tP$ (`x @ P` if you use NumPy arrays). Return the scores after `iterations` updates. This repeated update is called **power iteration**.

In the final cell, the transition matrix you constructed and the `pagerank`-function are used to rank the Wikipedia pages.

In [8]:
import json

with open("painters.json", encoding="utf-8") as f:
    painters = json.load(f)

pages = painters["names"]
n = len(pages)
A = [[0] * n for _ in range(n)]
for source, target in painters["links"]:
    A[source][target] = 1

x0 = [1 / n] * n


In [27]:
# TODO: Construct the transition matrix and compute PageRank
#Virker veldig rett fram, men tror jeg leser slidsa uansett.
#Okay har gått gjennom slidsa.
import copy
def transition_matrix(A, alpha=0.15):
    n = len(A)
    trans_matrix = [[0.0] * n for _ in range(n)]

    for i, row in enumerate(A):
        n_out = sum(row) 
        for j, connection in enumerate(row):
            if n_out == 0:
                p = 1 / n
            elif connection == 1:
                p = 1 / n_out
            else:
                p = 0
            trans_matrix[i][j] = (1 - alpha) * p + alpha / n
    return trans_matrix

#Hold up, vi trenger jo alpha også når vi gjøre pagerank?? for å regne ut
#nye score??
def pagerank(P, x0, iterations=50, alpha=0.15):
    #Skal nå traversere,  x0 er start_score, P er page matrisen
    #x0 er en liste med start scor for hver page.

    x_current_score = x0
    for i in range(0, iterations):
        #Idéen med x_next_score er å bygge opp scoreene også kopiere til slutt.
        x_next_score = [0] * n

        for page_index, page in enumerate(P):
            x_next_score[page_index] += (alpha/n)

            for link_index, link_value in enumerate(page):
                x_next_score[link_index] += link_value * x_current_score[page_index] *(1-alpha)

        x_current_score = copy.deepcopy(x_next_score)

    return x_current_score

trans_matrix = transition_matrix(A)
x_final_score = pagerank(trans_matrix, x0)
print(x_final_score)

#Sanity check x_final_score adder up til 1
print(sum(x_final_score))

[0.06922758911646197, 0.07172986418788133, 0.10424563083314416, 0.10960769757126916, 0.0954761652865496, 0.04850284176222936, 0.03103159071847917, 0.05671335116682733, 0.05510448494174647, 0.12689356576943447, 0.058704745511701156, 0.05665937935553197, 0.08507150306026463, 0.03103159071847917]
1.0


In [28]:
P = transition_matrix(A, alpha=0.15)
scores = pagerank(P, x0)
incoming_links = [sum(A[i][j] for i in range(n)) for j in range(n)]
for i in sorted(range(n), key=lambda i: scores[i], reverse=True):
    print(f"{pages[i]:24} PageRank: {scores[i]:.5f}   Incoming links: {incoming_links[i]}")


Leonardo da Vinci        PageRank: 0.12689   Incoming links: 3
Edouard Manet            PageRank: 0.10961   Incoming links: 6
Michel Angelo            PageRank: 0.10425   Incoming links: 4
Peter Paul Rubens        PageRank: 0.09548   Incoming links: 4
Pierre-Auguste Renoir    PageRank: 0.08507   Incoming links: 5
Claude Monet             PageRank: 0.07173   Incoming links: 5
Pablo Picasso            PageRank: 0.06923   Incoming links: 5
Henri Matisse            PageRank: 0.05870   Incoming links: 3
Edgar Degas              PageRank: 0.05671   Incoming links: 3
Paul Cezanne             PageRank: 0.05666   Incoming links: 4
Vincent van Gogh         PageRank: 0.05510   Incoming links: 4
Rembrandt                PageRank: 0.04850   Incoming links: 2
Gustav Klimt             PageRank: 0.03103   Incoming links: 1
Egon Schiele             PageRank: 0.03103   Incoming links: 1


## 4.3 Conceptual questions

Answer briefly, using your results where relevant.

1. Why can the biword index return extra documents for longer phrases? How does the positional index avoid these matches?

The biword index can return extra documents for longer phrases because every word doesn't have to be in a consecutive index. The positional index avoid these matches because it checks if every token in search phrase appears in consecutive order.

2. Does the painter with the highest PageRank also have the most incoming links? Explain what PageRank captures beyond counting incoming links.

Leonarda da VinCI has the highest page rank, but he does not the highest incoming links. This is because PageRank considers the ranking of the pages that have inlinks to Leonardo Da Vinci. In this case Michelangela and Peter Paul has a high score themselves and have inlinks to Leonardo.

3. Why does PageRank include teleportation?

PageRank includes teleportation to make sure we end up with a stable result, instead of a result that ocilates betwwen some states back and forth.

4. Why might a crawler fetch a lower-priority URL from one host before a higher-priority URL from another host?

A crawler might fetch a low priority URL before a higher priority URL if the host from the high priority URL is unavailable. Also if the URL has a politeness request that ask you to limit the rate at which you make requests.